# PlantVision 1.1 — Houseplant disease pipeline

Stage 1: YOLOv8n one-class leaf detector. Stage 2: EfficientNet-B0 @ 288. 10 классов: `anthracnose`, `bacterial_leaf_spot`, `gray_mold`, `healthy`, `iron_chlorosis`, `late_blight`, `mosaic_virus`, `not_a_plant`, `powdery_mildew`, `sunburn`.

Целевая метрика — **disease macro-F1** (8 болезней, без `healthy` / `not_a_plant`). Healthy bias monitor каждую эпоху (alarm >2.0). Class-balanced CE с asymmetric down-weighting healthy ×0.25 + jpeg/blur degradation на 50% HP-кропов. Реальные пользовательские фото из `RAW/real_world/<class>/` (или RAR через `REAL_WORLD_DATASET_URL`) идут в train.

Бюджет: A100-40 GB ≈ 35-45 мин end-to-end. Disk ≥30 GB. Run All → артефакты в `out/*` → копировать в `server/models/`.

## 1. Зависимости и окружение

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'ultralytics>=8.2.0', 'kaggle', 'onnx>=1.16', 'onnxruntime>=1.17',
    'onnxsim>=0.4', 'requests', 'scikit-learn', 'matplotlib', 'gdown>=5.1'])

if Path('/etc/debian_version').exists() and os.geteuid() == 0:
    pkgs = [p for p in ('aria2', 'unrar') if not shutil.which(p.replace('aria2', 'aria2c'))]
    if pkgs:
        subprocess.run(['apt-get', 'install', '-y', '-qq'] + pkgs,
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import torch, torchvision, ultralytics, onnxruntime as ort

if 'google.colab' in sys.modules or (Path('/content').is_dir() and os.access('/content', os.W_OK)):
    ENV = 'colab'
elif Path('/kaggle/working').is_dir() and os.access('/kaggle/working', os.W_OK):
    ENV = 'kaggle'
else:
    ENV = 'local'

_DEFAULT = {'colab': '/content/plantvision', 'kaggle': '/kaggle/working/plantvision',
            'local': str(Path.cwd() / 'plantvision_workdir')}[ENV]
WORKDIR = Path(os.getenv('PLANTVISION_WORKDIR', _DEFAULT)).expanduser().resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

_t = min(max(1, os.cpu_count() or 1), 8)
for k in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(k, str(_t))
torch.set_num_threads(_t)
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
IS_A100  = 'A100' in GPU_NAME

print(f'PyTorch {torch.__version__} | TV {torchvision.__version__} | UL {ultralytics.__version__} | ORT {ort.__version__}')
print(f'Device: {device} | Env: {ENV} | Workdir: {WORKDIR}')
if torch.cuda.is_available():
    print(f'GPU: {GPU_NAME} ({VRAM_GB:.1f} GB) | bf16={HAS_BF16} | A100={IS_A100}')
disk_free = shutil.disk_usage(str(WORKDIR)).free / 1e9
print(f'Disk free: {disk_free:.1f} GB')
if disk_free < 25:
    print(f'⚠ <25 GB свободно — pipeline может упасть.')


## 2. Загрузка датасетов

PlantDoc OD (детектор), PlantVillage sparse (PV-классы), iNaturalist (wild), COCO val2017 (NAP), Houseplant Species Kaggle (healthy через детектор).

Kaggle creds: `~/.kaggle/kaggle.json`, либо `KAGGLE_USERNAME` + `KAGGLE_KEY` env-vars, либо в Colab — file picker.

In [ ]:
import subprocess, time, json, re, gc, hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from requests.adapters import HTTPAdapter

RAW = WORKDIR / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

def _has(cmd):
    return subprocess.run(['which', cmd], stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL).returncode == 0

def _link_or_copy(src, dst):
    if dst.exists(): return
    try: os.link(src, dst)
    except (OSError, FileNotFoundError): shutil.copy2(src, dst)

def _download(url, out):
    if _has('aria2c'):
        subprocess.check_call(['aria2c', '-x', '16', '-s', '16', '-q',
            '--allow-overwrite=true', '--auto-file-renaming=false',
            '-d', str(out.parent), '-o', out.name, url])
    else:
        subprocess.check_call(['wget', '-q', '-O', str(out), url])

def _unzip(src, dst, only=None):
    cmd = ['unzip', '-q', '-o', str(src)]
    if only: cmd.append(only)
    cmd.extend(['-d', str(dst)])
    subprocess.check_call(cmd)

# PlantDoc OD
plantdoc_dir = RAW / 'plantdoc_od'
def fetch_plantdoc():
    if plantdoc_dir.exists():
        return f'PlantDoc cached ({sum(1 for _ in plantdoc_dir.rglob("*.jpg"))} jpg).'
    subprocess.check_call(['git', 'clone', '--depth', '1', '-q',
        'https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git',
        str(plantdoc_dir)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return f'PlantDoc OD: {sum(1 for _ in plantdoc_dir.rglob("*.jpg"))} jpg.'

# PlantVillage sparse
PV_FOLDERS = {
    'powdery_mildew': ['raw/color/Squash___Powdery_mildew',
                       'raw/color/Cherry_(including_sour)___Powdery_mildew'],
    'iron_chlorosis': ['raw/color/Tomato___Tomato_Yellow_Leaf_Curl_Virus'],
    'late_blight':    ['raw/color/Tomato___Late_blight', 'raw/color/Potato___Late_blight'],
    'sunburn':        ['raw/color/Strawberry___Leaf_scorch'],
    'bacterial_leaf_spot': ['raw/color/Tomato___Bacterial_spot',
                            'raw/color/Pepper,_bell___Bacterial_spot'],
    'mosaic_virus':   ['raw/color/Tomato___Tomato_mosaic_virus'],
}
pv_root = RAW / 'plantvillage_repo'
pv_extracted = RAW / 'plantvillage_extracted'
def fetch_plantvillage():
    if pv_extracted.exists() and any(pv_extracted.iterdir()):
        return 'PV cached.'
    pv_extracted.mkdir(exist_ok=True)
    if not pv_root.exists():
        subprocess.check_call(['git', 'clone', '--filter=blob:none', '--no-checkout',
            '--depth', '1', '-q', 'https://github.com/spMohanty/PlantVillage-Dataset.git',
            str(pv_root)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(['git', '-C', str(pv_root), 'sparse-checkout', 'init', '--cone'],
                          stdout=subprocess.DEVNULL)
    paths = [p for v in PV_FOLDERS.values() for p in v]
    subprocess.check_call(['git', '-C', str(pv_root), 'sparse-checkout', 'set'] + paths,
                          stdout=subprocess.DEVNULL)
    subprocess.check_call(['git', '-C', str(pv_root), 'checkout', '-q'], stdout=subprocess.DEVNULL)
    total = 0
    for cls, paths in PV_FOLDERS.items():
        out = pv_extracted / cls; out.mkdir(exist_ok=True)
        for p in paths:
            src = pv_root / p
            if not src.exists(): continue
            for f in src.iterdir():
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    _link_or_copy(f, out / f.name); total += 1
    return f'PV: {total} files linked.'

# COCO val2017
coco_dir = RAW / 'coco_val2017'
def fetch_coco_images():
    if coco_dir.exists() and any(coco_dir.glob('*.jpg')):
        return f'COCO cached ({sum(1 for _ in coco_dir.glob("*.jpg"))} jpg).'
    z = RAW / 'coco_val2017.zip'
    _download('http://images.cocodataset.org/zips/val2017.zip', z)
    _unzip(z, RAW)
    extracted = RAW / 'val2017'
    if extracted.exists(): extracted.rename(coco_dir)
    z.unlink(missing_ok=True)
    return f'COCO val2017: {sum(1 for _ in coco_dir.glob("*.jpg"))} jpg.'

coco_ann = RAW / 'annotations' / 'instances_val2017.json'
def fetch_coco_annotations():
    if coco_ann.exists(): return 'COCO ann cached.'
    z = RAW / 'coco_ann.zip'
    _download('http://images.cocodataset.org/annotations/annotations_trainval2017.zip', z)
    _unzip(z, RAW, only='annotations/instances_val2017.json')
    z.unlink(missing_ok=True)
    return 'COCO ann ok.'

# iNaturalist
INAT_TAXA = {
    'powdery_mildew':   [(55524, 'Erysiphales'), (55525, 'Erysiphaceae'), (334961, 'Podosphaera')],
    'gray_mold':        [(324263, 'Botrytis cinerea'), (324264, 'Botrytis'), (204631, 'Sclerotiniaceae')],
    'late_blight':      [(199345, 'P. cinnamomi'), (48802, 'P. ramorum'),
                         (53860, 'P. infestans'), (52594, 'Phytophthora')],
    'anthracnose':      [(323958, 'Colletotrichum'), (381773, 'C. trichellum'),
                         (524080, 'Glomerella'), (1109193, 'C. gloeosporioides'),
                         (358169, 'C. acutatum')],
    'bacterial_leaf_spot': [(337405, 'P. syringae'), (132056, 'Xanthomonas'), (384743, 'X. campestris')],
    'mosaic_virus':     [(916920, 'Tobamovirus'), (1027410, 'TMV'),
                         (130930, 'CMV'), (1183891, 'Tobamoviridae'),
                         (915687, 'Begomovirus')],
}
TARGET_PER_TAXON = 800
INAT_BASE = 'https://api.inaturalist.org/v1/observations'
_HTTP = requests.Session()
_HTTP.mount('https://', HTTPAdapter(pool_connections=20, pool_maxsize=20))
_HTTP.mount('http://',  HTTPAdapter(pool_connections=20, pool_maxsize=20))
_IMG_MAGIC = (b'\xff\xd8\xff', b'\x89PNG\r\n\x1a\n', b'GIF87a', b'GIF89a', b'RIFF')

def _inat_dl(url, dst):
    if dst.exists(): return True
    try: r = _HTTP.get(url, timeout=15)
    except Exception: return False
    if r.status_code != 200 or not (4096 < len(r.content) <= 8 * 1024 * 1024): return False
    if not any(r.content.startswith(m) for m in _IMG_MAGIC): return False
    try: dst.write_bytes(r.content); return True
    except Exception: return False

def _inat_taxon(tid, target, out_dir, saved):
    page = 1
    with ThreadPoolExecutor(max_workers=16) as pool:
        while saved < target and page <= 30:
            q = {'taxon_id': tid, 'photo_license': 'cc0,cc-by,cc-by-sa',
                 'quality_grade': 'research', 'identified': 'true',
                 'per_page': 200, 'page': page}
            try:
                r = _HTTP.get(INAT_BASE, params=q, timeout=30)
                if r.status_code != 200: break
                results = r.json().get('results', [])
            except Exception: break
            if not results: break
            jobs = []
            for obs in results:
                for ph in obs.get('photos', []):
                    if (ph.get('license_code') or '').lower() not in ('cc0', 'cc-by', 'cc-by-sa'):
                        continue
                    url = (ph.get('url') or '').replace('/square.', '/medium.').replace('/small.', '/medium.')
                    if not url: continue
                    m = re.search(r'\.(jpe?g|png)(?:\?|$)', url, re.IGNORECASE)
                    ext = '.' + m.group(1).lower() if m else '.jpg'
                    jobs.append((url, out_dir / f"t{tid}_{obs['id']}_{ph['id']}{ext}"))
                    if len(jobs) + saved >= target * 2: break
                if len(jobs) + saved >= target * 2: break
            for fut in as_completed([pool.submit(_inat_dl, u, f) for u, f in jobs]):
                if fut.result(): saved += 1
                if saved >= target: break
            page += 1
            time.sleep(1.1)
    return saved

def fetch_inaturalist():
    inat_root.mkdir(exist_ok=True)
    msgs = []
    for key, fbs in INAT_TAXA.items():
        out = inat_root / key
        out.mkdir(parents=True, exist_ok=True)
        have = sum(1 for f in out.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        if have >= TARGET_PER_TAXON:
            msgs.append(f'iNat {key}: {have} [cached]'); continue
        saved, src = have, []
        for tid, lbl in fbs:
            before = saved
            saved = _inat_taxon(tid, TARGET_PER_TAXON, out, saved)
            if saved > before: src.append(f'{lbl}+{saved-before}')
            if saved >= TARGET_PER_TAXON: break
        msgs.append(f'iNat {key}: {saved} [{",".join(src) or "none"}]')
    return '\n  '.join(msgs)

inat_root = RAW / 'inaturalist'

JOBS = [('plantdoc', fetch_plantdoc), ('plantvillage', fetch_plantvillage),
        ('coco_images', fetch_coco_images), ('coco_ann', fetch_coco_annotations),
        ('inaturalist', fetch_inaturalist)]
print(f'Fetching {len(JOBS)} sources (aria2c={"yes" if _has("aria2c") else "no"})…')
t0 = time.time()
with ThreadPoolExecutor(max_workers=len(JOBS)) as pool:
    futures = {pool.submit(fn): name for name, fn in JOBS}
    for fut in as_completed(futures):
        name = futures[fut]
        try: print(f'  ✓ {name}: {fut.result()}')
        except Exception as e: print(f'  ✗ {name}: {e}')
print(f'Download: {time.time()-t0:.1f}s')

# Houseplant Species (Kaggle)
hp_dir = RAW / 'houseplant'
def _kaggle_creds():
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        try: kj.chmod(0o600)
        except Exception: pass
        return True
    user, key = os.getenv('KAGGLE_USERNAME', '').strip(), os.getenv('KAGGLE_KEY', '').strip()
    if user and key:
        kj.parent.mkdir(parents=True, exist_ok=True)
        kj.write_text(json.dumps({'username': user, 'key': key}), encoding='utf-8')
        try: kj.chmod(0o600)
        except Exception: pass
        return True
    if ENV == 'colab':
        try:
            from google.colab import files  # type: ignore
            print('▶ Загрузи kaggle.json (kaggle.com/settings → API → Create New Token)')
            up = files.upload()
            picked = next((it for it in up.items() if it[0].lower().endswith('.json')),
                          next(iter(up.items()), None))
            if not picked: return False
            fname, content = picked
            try: parsed = json.loads(content.decode('utf-8'))
            except Exception: return False
            if 'username' not in parsed or 'key' not in parsed: return False
            kj.parent.mkdir(parents=True, exist_ok=True)
            kj.write_bytes(content)
            try: kj.chmod(0o600)
            except Exception: pass
            return True
        except Exception: return False
    return False

if not hp_dir.exists():
    if _kaggle_creds():
        try:
            subprocess.check_call(['kaggle', 'datasets', 'download', '-d',
                'kacpergregorowicz/house-plant-species', '-p', str(hp_dir), '--unzip', '-q'])
        except Exception as e:
            print(f'Kaggle HP download failed: {e}')
    else:
        print('⚠ Kaggle creds not found — `healthy` будет дропнут на cell 10.')
n_hp = (sum(1 for _ in hp_dir.rglob('*.jpg')) + sum(1 for _ in hp_dir.rglob('*.jpeg'))
        if hp_dir.exists() else 0)
print(f'Houseplant: {n_hp} images')

# Real-world пользовательские фото. Архив с структурой <class>/photo.jpg
# (например dataset.rar/healthy/, dataset.rar/anthracnose/).
# Можно положить файлы вручную в RAW/real_world/<class>/ — тогда URL не нужен.
real_world_dir = RAW / 'real_world'
REAL_WORLD_URL = (os.getenv('REAL_WORLD_DATASET_URL', '').strip()
                  or os.getenv('REAL_WORLD_RAR_URL', '').strip())

def _gdrive_id(url):
    m = re.search(r'(?:/d/|id=|/file/d/)([A-Za-z0-9_-]{20,})', url)
    return m.group(1) if m else None

def _flatten_real_world_root():
    # Архив часто заворачивает всё в одну папку (dataset/healthy/...).
    # Если в real_world_dir одна подпапка, и в ней лежат подпапки-классы — поднимем их наверх.
    sub = [d for d in real_world_dir.iterdir() if d.is_dir()]
    if len(sub) == 1:
        inner = [d for d in sub[0].iterdir() if d.is_dir()]
        if inner:
            for d in inner:
                shutil.move(str(d), str(real_world_dir / d.name))
            try: sub[0].rmdir()
            except OSError: pass

def fetch_real_world():
    if real_world_dir.exists() and any(real_world_dir.rglob('*.jpg')):
        _flatten_real_world_root()
        n = sum(1 for p in real_world_dir.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        cls = sorted({d.name for d in real_world_dir.iterdir() if d.is_dir()})
        return f'real_world cached ({n} images, classes={cls})'
    if not REAL_WORLD_URL:
        return 'real_world skipped (set REAL_WORLD_DATASET_URL or place files in RAW/real_world/<class>/)'
    rar_path = RAW / 'real_world.rar'
    fid = _gdrive_id(REAL_WORLD_URL)
    try:
        if fid:
            subprocess.check_call([sys.executable, '-m', 'gdown', '--id', fid,
                                   '-O', str(rar_path), '--quiet'])
        else:
            subprocess.check_call(['wget', '-q', '-O', str(rar_path), REAL_WORLD_URL])
    except Exception as e:
        return f'real_world download failed: {e}'
    if not shutil.which('unrar'):
        return 'real_world: unrar not installed (re-run cell 2 with sudo)'
    real_world_dir.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.check_call(['unrar', 'x', '-y', '-inul', str(rar_path),
                               str(real_world_dir) + '/'])
    except Exception as e:
        return f'real_world extract failed: {e}'
    _flatten_real_world_root()
    rar_path.unlink(missing_ok=True)
    n = sum(1 for p in real_world_dir.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    cls = sorted({d.name for d in real_world_dir.iterdir() if d.is_dir()})
    return f'real_world: {n} images, classes={cls}'

print(fetch_real_world())

du = shutil.disk_usage(str(WORKDIR))
print(f'Disk: used {du.used/1e9:.1f} GB, free {du.free/1e9:.1f} GB')


## 3. YOLO-датасет: один класс `leaf` (PlantDoc 85/15)

In [ ]:
import xml.etree.ElementTree as ET
import random

YOLO_DIR = WORKDIR / 'detector'
if YOLO_DIR.exists(): shutil.rmtree(YOLO_DIR)
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (YOLO_DIR / sub).mkdir(parents=True, exist_ok=True)

random.seed(42)
pairs = []
for split_name in ('TRAIN', 'TEST'):
    sd = plantdoc_dir / split_name
    if not sd.exists(): continue
    for xml in sd.rglob('*.xml'):
        for ext in ('.jpg', '.JPG', '.jpeg', '.png'):
            img = xml.with_suffix(ext)
            if img.exists(): pairs.append((img, xml)); break
print(f'PlantDoc XML pairs: {len(pairs)}')
random.shuffle(pairs)
cut = int(len(pairs) * 0.85)
train_pairs, val_pairs = pairs[:cut], pairs[cut:]

def write_yolo(pair_list, split):
    kept = 0
    for img, xml in pair_list:
        try: root = ET.parse(xml).getroot()
        except ET.ParseError: continue
        size = root.find('size')
        if size is None: continue
        W = float(size.findtext('width') or 0); H = float(size.findtext('height') or 0)
        if W <= 0 or H <= 0: continue
        lines = []
        for obj in root.findall('object'):
            if 'leaf' not in (obj.findtext('name') or '').lower(): continue
            bb = obj.find('bndbox')
            if bb is None: continue
            x1 = float(bb.findtext('xmin') or 0); y1 = float(bb.findtext('ymin') or 0)
            x2 = float(bb.findtext('xmax') or 0); y2 = float(bb.findtext('ymax') or 0)
            if x2 <= x1 or y2 <= y1: continue
            cx = min(max(((x1+x2)/2)/W, 0), 1); cy = min(max(((y1+y2)/2)/H, 0), 1)
            bw = min(max((x2-x1)/W, 0.001), 1); bh = min(max((y2-y1)/H, 0.001), 1)
            lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        if not lines: continue
        _link_or_copy(img, YOLO_DIR / f'images/{split}' / img.name)
        (YOLO_DIR / f'labels/{split}' / (img.stem + '.txt')).write_text('\n'.join(lines), encoding='utf-8')
        kept += 1
    return kept

n_tr = write_yolo(train_pairs, 'train')
n_vl = write_yolo(val_pairs, 'val')
print(f'YOLO train={n_tr}  val={n_vl}')

data_yaml = YOLO_DIR / 'data.yaml'
data_yaml.write_text(
    f'path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [leaf]\n',
    encoding='utf-8')


## 4. Тренировка YOLOv8n (Stage 1)

60 эпох, patience=12, AMP, cos_lr, mosaic=0.5. Экспорт ONNX FP32 opset=13.

In [ ]:
from ultralytics import YOLO

det_model = YOLO('yolov8n.pt')
_yolo_batch = 64 if VRAM_GB >= 24 else 32
det_model.train(
    data=str(data_yaml), epochs=60, imgsz=640, batch=_yolo_batch, patience=12,
    cos_lr=True, amp=True, cache='ram' if VRAM_GB >= 24 else 'disk',
    workers=8, save_period=-1,
    close_mosaic=10, mosaic=0.5, mixup=0.1, copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5,
    project=str(WORKDIR / 'yolo_runs'), name='detector', exist_ok=True, verbose=True,
)

best_pt = WORKDIR / 'yolo_runs' / 'detector' / 'weights' / 'best.pt'
assert best_pt.exists(), f'YOLO best.pt missing: {best_pt}'
MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)
shutil.copy2(best_pt, MODELS_OUT / 'detector.pt')

det_model.export(format='onnx', imgsz=640, opset=13, simplify=True, dynamic=False)
onnx_src = best_pt.with_suffix('.onnx')
if onnx_src.exists():
    shutil.copy2(onnx_src, MODELS_OUT / 'detector.onnx')

metrics = det_model.val(data=str(data_yaml), imgsz=640, plots=False, verbose=False)
print(f'mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}')

del det_model
gc.collect()
import torch
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 5. Классификаторный датасет (Stage 2)

10 классов: PV clean + iNat wild + HP→healthy через детектор + COCO→not_a_plant + real-world пользовательские фото. SHA-256 dedup, group-aware split, EXIF-strip для iNat, hardlink для PV.

**HP→healthy:** target **500** фото, **CROPS_PER_PHOTO=1** (только самый большой bbox). На **50%** HP-кропов накладывается жёсткая аугментация (blur 1-2.5 + JPEG q=15-30) — снимает studio-quality перевес.

**COCO→not_a_plant:** cap **800** (раньше 1800 размывал disease feature-границы).

**iNat fallback** для слабых классов расширен: `anthracnose` +Glomerella +C.gloeosporioides +C.acutatum; `mosaic_virus` +Tobamoviridae +Begomovirus.

**Real-world фото** (`RAW/real_world/<class>/*.jpg`): сплит 80/20 поверх обычного train/val. Real-world healthy capped at **30** — иначе живые 60-70% healthy воссоздают prod-bias.

`<60 train` → класс дропается; `healthy` дроп → hard-fail.

In [ ]:
import json, re, hashlib
from PIL import Image, ImageFilter
import numpy as np
import torch
from ultralytics import YOLO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)
CLS_DIR = WORKDIR / 'classifier'
if CLS_DIR.exists(): shutil.rmtree(CLS_DIR)

TARGET_CLASSES = ['anthracnose', 'bacterial_leaf_spot', 'gray_mold', 'healthy',
                  'iron_chlorosis', 'late_blight', 'mosaic_virus', 'not_a_plant',
                  'powdery_mildew', 'sunburn']
random.seed(42)
VAL_RATIO = 0.15
MIN_VAL_PER_CLASS = 8
MIN_TRAIN_PER_CLASS = 60

for split in ('train', 'val'):
    for cls in TARGET_CLASSES:
        (CLS_DIR / split / cls).mkdir(parents=True, exist_ok=True)

_SEEN = set()
def _file_hash(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def _pil_hash(img):
    arr = np.asarray(img.convert('RGB').resize((128, 128), Image.BILINEAR), dtype=np.uint8)
    return hashlib.sha256(arr.tobytes()).hexdigest()

def _add_file(pool_list, path, group):
    try: h = _file_hash(path)
    except Exception: return False
    if h in _SEEN: return False
    _SEEN.add(h); pool_list.append(('file', path, group)); return True

def _add_pil_path(pool_list, pil_img, save_path, group, hard_aug=False):
    try: h = _pil_hash(pil_img)
    except Exception: return False
    if h in _SEEN: return False
    try:
        if hard_aug:
            # Blur + low-quality JPEG: снимает "studio quality" перевес healthy.
            pil_img = pil_img.filter(ImageFilter.BoxBlur(random.uniform(1.0, 2.5)))
            pil_img.save(save_path, 'JPEG', quality=random.randint(15, 30), optimize=True)
        else:
            pil_img.save(save_path, 'JPEG', quality=92, optimize=True)
    except Exception: return False
    _SEEN.add(h); pool_list.append(('file', save_path, group)); return True

pool = {c: [] for c in TARGET_CLASSES}

# PV
PV_TO_CLASS = {k: k for k in ('powdery_mildew', 'iron_chlorosis', 'late_blight',
                              'sunburn', 'bacterial_leaf_spot', 'mosaic_virus')}
PV_CAP = {'powdery_mildew': 500, 'iron_chlorosis': 500, 'late_blight': 800,
          'sunburn': 500, 'bacterial_leaf_spot': 700, 'mosaic_virus': 700}
for pv_cls, dst in PV_TO_CLASS.items():
    sd = pv_extracted / pv_cls
    if not sd.exists(): continue
    imgs = [p for p in sd.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    sel = imgs[:PV_CAP[pv_cls]]
    n = sum(1 for p in sel if _add_file(pool[dst], p, None))
    print(f'PV {pv_cls}: {n}/{len(sel)}')

# iNat
INAT_OTHER = {
    'powdery_mildew': ('powdery_mildew', 600), 'gray_mold': ('gray_mold', 700),
    'late_blight': ('late_blight', 400), 'anthracnose': ('anthracnose', 700),
    'bacterial_leaf_spot': ('bacterial_leaf_spot', 500),
    'mosaic_virus': ('mosaic_virus', 500),
}
def _inat_group(p):
    m = re.match(r't(\d+)_(\d+)_\d+', p.stem)
    return f'inat_obs_{m.group(2)}' if m else None

for key, (dst, cap) in INAT_OTHER.items():
    sd = inat_root / key
    if not sd.exists(): continue
    imgs = [p for p in sd.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    sel = imgs[:cap]
    n = sum(1 for p in sel if _add_file(pool[dst], p, _inat_group(p)))
    print(f'iNat {key}: {n}/{len(sel)}')

# HP → healthy via detector. CROPS_PER_PHOTO=1 (только самый большой bbox)
# чтобы убрать корреляцию между разными кропами одного листа.
# 50% кропов получают жёсткую аугментацию (blur + low-quality JPEG) —
# снимает studio-quality bias, главный драйвер healthy bias в проде.
HP_TARGET = 500
CROPS_PER_PHOTO = 1
HP_HARD_AUG_RATIO = 0.50
DETECT_BATCH = 64 if VRAM_GB >= 24 else 16
HP_CROPS_DIR = RAW / 'hp_crops'
HP_CROPS_DIR.mkdir(exist_ok=True)

def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0: return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def _nms(boxes, k, thr=0.5):
    boxes = sorted(boxes, key=lambda b: -((b[2]-b[0])*(b[3]-b[1])))
    kept = []
    for b in boxes:
        if any(_iou(b, x) > thr for x in kept): continue
        kept.append(b)
        if len(kept) >= k: break
    return kept

if hp_dir.exists():
    det = YOLO(str(MODELS_OUT / 'detector.pt'))
    hp_imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
        hp_imgs.extend(hp_dir.rglob(ext))
    random.shuffle(hp_imgs)
    saved = 0
    crop_idx = 0
    i = 0
    while i < len(hp_imgs) and saved < HP_TARGET:
        chunk = hp_imgs[i:i + DETECT_BATCH]; i += DETECT_BATCH
        pils = []
        for p in chunk:
            try: pils.append(Image.open(p).convert('RGB'))
            except Exception: pils.append(None)
        valid = [(p, pi) for p, pi in zip(chunk, pils) if pi is not None]
        if not valid: continue
        arrs = [np.array(pi) for _, pi in valid]
        try: results = det.predict(arrs, imgsz=640, conf=0.4, verbose=False, device=device)
        except Exception: continue
        for (path, pil), res in zip(valid, results):
            if saved >= HP_TARGET: break
            if res.boxes is None or len(res.boxes) == 0: continue
            boxes = res.boxes.xyxy.cpu().numpy().tolist()
            W, H = pil.size
            kept = _nms(boxes, CROPS_PER_PHOTO, 0.5)
            grp = f'hp_{path.stem}'
            for b in kept:
                x1, y1, x2, y2 = (int(round(v)) for v in b)
                px, py = int((x2-x1)*0.1), int((y2-y1)*0.1)
                x1, y1 = max(0, x1-px), max(0, y1-py)
                x2, y2 = min(W, x2+px), min(H, y2+py)
                if x2-x1 < 32 or y2-y1 < 32: continue
                crop = pil.crop((x1, y1, x2, y2))
                hard = random.random() < HP_HARD_AUG_RATIO
                dst = HP_CROPS_DIR / f'hp_{crop_idx:06d}.jpg'
                if _add_pil_path(pool['healthy'], crop, dst, grp, hard_aug=hard):
                    saved += 1; crop_idx += 1
                if saved >= HP_TARGET: break
        for _, pi in valid:
            try: pi.close()
            except Exception: pass
    print(f'HP → healthy: {saved} crops')
    del det, pils
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# COCO → not_a_plant
PLANT_COCO_IDS = {64, 86, 56, 50, 51, 52, 53, 54, 55, 57, 58, 59, 60, 61}
banned = set()
if coco_ann.exists():
    meta = json.loads(coco_ann.read_text(encoding='utf-8'))
    for a in meta.get('annotations', []):
        if a.get('category_id') in PLANT_COCO_IDS:
            banned.add(a['image_id'])
    print(f'COCO ban: {len(banned)}')
    del meta; gc.collect()

def _cid(p):
    try: return int(p.stem)
    except ValueError: return -1

coco_imgs = [p for p in coco_dir.glob('*.jpg') if _cid(p) not in banned]
random.shuffle(coco_imgs)
# 1800 не_растения сильно перевешивали disease классы (2-3×) →
# сэмплер сглаживал, но это размывало feature-границы. 800 достаточно.
COCO_CAP = 800
n = sum(1 for p in coco_imgs[:COCO_CAP] if _add_file(pool['not_a_plant'], p, None))
print(f'COCO → NAP: {n}')

# Stratified group-aware split + materialize
stats = {c: {'train': 0, 'val': 0} for c in TARGET_CLASSES}

def _strip_save(src_path, dst):
    try:
        with Image.open(src_path) as im:
            im.load()
            im.convert('RGB').save(dst, 'JPEG', quality=92, optimize=True, exif=b'')
        return True
    except Exception: return False

def _materialize(items, cls, split):
    out = CLS_DIR / split / cls
    for kind, payload, _ in items:
        idx = stats[cls][split]
        # iNat → re-encode + EXIF strip (user GPS / camera serial).
        # PV / COCO / HP crops → hardlink (no EXIF concern).
        if 'inaturalist' in str(payload).lower():
            dst = out / f'{cls}_{idx:06d}.jpg'
            if _strip_save(payload, dst): stats[cls][split] += 1
        else:
            dst = out / f'{cls}_{idx:06d}{payload.suffix.lower() or ".jpg"}'
            try: _link_or_copy(payload, dst); stats[cls][split] += 1
            except Exception: pass

def _split(items, vr, mn):
    groups = {}
    for i, it in enumerate(items):
        key = it[2] if (len(it) >= 3 and it[2] is not None) else f'_solo_{i}'
        groups.setdefault(key, []).append(it)
    keys = list(groups.keys()); random.shuffle(keys)
    n_total = len(items)
    target = max(int(round(n_total * vr)), mn if n_total >= mn * 2 else 1)
    target = min(target, n_total - 1)
    val_items, train_items = [], []
    for k in keys:
        (val_items if len(val_items) < target else train_items).extend(groups[k])
    return train_items, val_items

for cls, items in pool.items():
    if not items: continue
    tr, vl = _split(items, VAL_RATIO, MIN_VAL_PER_CLASS)
    _materialize(tr, cls, 'train')
    _materialize(vl, cls, 'val')

del pool; gc.collect()

dropped = []
for cls in list(TARGET_CLASSES):
    if stats[cls]['train'] < MIN_TRAIN_PER_CLASS:
        dropped.append(cls)
        for split in ('train', 'val'):
            d = CLS_DIR / split / cls
            if d.exists(): shutil.rmtree(d)
        TARGET_CLASSES.remove(cls); stats.pop(cls, None)

# ── Real-world пользовательские фото поверх обычного train/val ──
# 80/20 per class. Дедупаются глобальным SHA-256, материализуются напрямую
# (не идут через pool/sampler). Real-world healthy НЕ получает hard-aug —
# это эталон. Healthy кэп (RW_HEALTHY_CAP) ограничивает количество живого
# healthy чтобы не воссоздать тот же bias, который мы пытаемся побороть:
# в типичном live датасете healthy=60-70%, наивная подгрузка усилит bias.
RW_VAL_RATIO = 0.20
RW_HEALTHY_CAP = 30
rw_added = {c: {'train': 0, 'val': 0} for c in TARGET_CLASSES}
if real_world_dir.exists():
    for cls in TARGET_CLASSES:
        rw_cls = real_world_dir / cls
        if not rw_cls.is_dir(): continue
        photos = sorted(p for p in rw_cls.iterdir()
                        if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        if not photos: continue
        random.shuffle(photos)
        if cls == 'healthy' and len(photos) > RW_HEALTHY_CAP:
            photos = photos[:RW_HEALTHY_CAP]
        n_val = max(1, int(round(len(photos) * RW_VAL_RATIO))) if len(photos) >= 5 else 0
        val_photos, train_photos = photos[:n_val], photos[n_val:]
        for split, plist in (('train', train_photos), ('val', val_photos)):
            out = CLS_DIR / split / cls
            for p in plist:
                try:
                    h = _file_hash(p)
                    if h in _SEEN: continue
                    _SEEN.add(h)
                    idx = stats[cls][split]
                    dst = out / f'rw_{cls}_{idx:06d}{p.suffix.lower() or ".jpg"}'
                    _strip_save(p, dst)
                    stats[cls][split] += 1
                    rw_added[cls][split] += 1
                except Exception:
                    pass
    rw_total = sum(v['train'] + v['val'] for v in rw_added.values())
    if rw_total > 0:
        print(f'real_world: +{rw_total} photos materialised (healthy capped at {RW_HEALTHY_CAP}):')
        for c in TARGET_CLASSES:
            v = rw_added[c]
            if v['train'] + v['val']: print(f'  {c:<22} +{v["train"]} train, +{v["val"]} val')
else:
    print('real_world: directory absent → skipped')

print('\nClass distribution (post dedup + split + real-world):')
for cls in TARGET_CLASSES:
    print(f'  {cls:<22} train={stats[cls]["train"]:>4} val={stats[cls]["val"]:>4}')
tt = sum(s["train"] for s in stats.values())
tv = sum(s["val"] for s in stats.values())
print(f'Total: {tt} train / {tv} val ({len(TARGET_CLASSES)} classes)')

if 'healthy' in dropped:
    raise RuntimeError("'healthy' dropped (<60 train). Server depends on it. Check Kaggle creds in cell 4.")
if dropped:
    print(f'⚠ Dropped: {dropped}')

dis_counts = [(c, stats[c]['train']) for c in TARGET_CLASSES if c not in ('healthy', 'not_a_plant')]
if dis_counts:
    mn = min(c for _, c in dis_counts); mx = max(c for _, c in dis_counts)
    if mx > 5 * max(mn, 1):
        print(f'⚠ Disease imbalance x{mx/max(mn,1):.1f} (max={mx}, min={mn}).')


## 6. Тренировка классификатора (EfficientNet-B0 @ 288)

**Primary metric:** disease macro-F1 (8 болезней, без `healthy`/`not_a_plant`). Top-1 accuracy маскирует проблему.

**Phases:** A — head only (12 ep, lr=1e-3·sqrt(B/96)); B — features.4-8 + classifier (30 ep, lr=5e-5·sqrt(B/96), ES patience=10 по disease F1).

**Loss:** class-balanced CE с inverse-frequency weights × asymmetric down-weighting healthy **×0.25** (жёсткая ручка против healthy bias). `label_smoothing=0.05`. **WeightedRandomSampler убран** — двойное взвешивание ломает обучение.

**Healthy bias monitor** каждую эпоху: `predicted_healthy / true_healthy` (val). Норма <1.5, alarm >2.0. Real-world healthy capped at 30 чтобы не воссоздать prod-bias.

EMA(0.999), CutMix p=0.4, MixUp p=0.2, grad clip 1.0, AMP (bf16 на A100, иначе fp16+GradScaler), `torch.compile` reduce-overhead, `channels_last`. Batch 192 на A100-40 / 256 на A100-80. Best checkpoint и ES — по disease F1. Бюджет ~35-45 мин на A100-40 end-to-end.

In [ ]:
import copy, io, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import classification_report, confusion_matrix, f1_score

RESUME_FROM_BACKUP = True
USE_TORCH_COMPILE  = True

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# EfficientNet-B0 нативная res 224, рекомендуемая high-res fine-tune 288.
# 288 даёт +1-2% top-1 vs 240/256 на disease detail, при цене ~25% больше
# времени per-epoch (всё ещё <40 min на A100-40 в полном цикле).
IMG_SIZE = 288
if VRAM_GB >= 70:
    BATCH_SIZE = 256
elif VRAM_GB >= 32:    # A100-40
    BATCH_SIZE = 192
elif VRAM_GB >= 20:
    BATCH_SIZE = 128
else:
    BATCH_SIZE = 64
NUM_WORKERS     = min(os.cpu_count() or 2, 8)
PREFETCH        = 4
PIN_MEMORY      = device.type == 'cuda'
PERSISTENT      = NUM_WORKERS > 0
_LR_SCALE       = math.sqrt(BATCH_SIZE / 96)
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.05
CUTMIX_PROB     = 0.4
MIXUP_PROB      = 0.2
MIXUP_ALPHA     = 0.2
EMA_DECAY       = 0.999
EARLY_STOP      = 10
HEALTHY_DOWNWEIGHT = 0.25   # asymmetric class-weight multiplier (0.25 жёстче 0.35);
                            # главный рычаг против healthy bias.
DISEASE_NAMES = {'anthracnose', 'bacterial_leaf_spot', 'gray_mold',
                 'iron_chlorosis', 'late_blight', 'mosaic_virus',
                 'powdery_mildew', 'sunburn'}


class RandomJPEG:
    def __init__(self, p=0.5, q=(50, 95)): self.p, self.qmn, self.qmx = p, q[0], q[1]
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = io.BytesIO()
        img.convert('RGB').save(buf, format='JPEG', quality=random.randint(self.qmn, self.qmx))
        buf.seek(0); out = Image.open(buf); out.load()
        return out.convert('RGB')


class WarmLight:
    def __init__(self, p=0.3): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        if img.mode != 'RGB': img = img.convert('RGB')
        r, g, b = img.split()
        rf = 1.05 + random.random() * 0.10
        bf = 0.85 - random.random() * 0.10
        r = r.point(lambda v, k=rf: min(255, int(v * k)))
        b = b.point(lambda v, k=bf: int(v * k))
        img = Image.merge('RGB', (r, g, b))
        return ImageEnhance.Contrast(img).enhance(0.85 + random.random() * 0.15)


class MotionBlur:
    def __init__(self, p=0.15): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        return img.filter(ImageFilter.BoxBlur(random.uniform(0.5, 1.5)))


class RandomShadow:
    def __init__(self, p=0.2): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        if img.mode != 'RGB': img = img.convert('RGB')
        w, h = img.size
        side = random.choice(['left', 'right', 'top', 'bottom'])
        intensity = random.uniform(0.5, 0.85)
        if side in ('left', 'right'):
            grad = np.linspace(intensity, 1.0, w) if side == 'left' else np.linspace(1.0, intensity, w)
            mask = np.tile(grad[None, :], (h, 1))
        else:
            grad = np.linspace(intensity, 1.0, h) if side == 'top' else np.linspace(1.0, intensity, h)
            mask = np.tile(grad[:, None], (1, w))
        arr = np.clip(np.array(img, dtype=np.float32) * mask[..., None], 0, 255).astype(np.uint8)
        return Image.fromarray(arr, 'RGB')


def make_train_tf(size):
    # Wide RandomResizedCrop scale (0.5-1.0): healthy training crops are
    # YOLO-cropped close-ups, disease photos are wide PV/iNat frames.
    # Aggressive scale jitter lets the model see both regimes during
    # training so inference (always YOLO close-up) can't latch onto
    # "small frame full of green = healthy".
    return transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.5, 1.0), ratio=(0.75, 1.33)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.1),
        WarmLight(0.3), RandomShadow(0.2), MotionBlur(0.15), RandomJPEG(0.5),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])

def make_val_tf(size):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


train_ds = datasets.ImageFolder(str(CLS_DIR / 'train'), transform=make_train_tf(IMG_SIZE))
val_ds   = datasets.ImageFolder(str(CLS_DIR / 'val'),   transform=make_val_tf(IMG_SIZE))
class_names = list(train_ds.classes)
num_classes = len(class_names)
not_a_plant_idx = class_names.index('not_a_plant') if 'not_a_plant' in class_names else -1
healthy_idx = class_names.index('healthy') if 'healthy' in class_names else -1
disease_idx = [i for i, n in enumerate(class_names) if n in DISEASE_NAMES]
print('Classes:', class_names)
print(f'healthy_idx={healthy_idx} | not_a_plant_idx={not_a_plant_idx} | disease_idx={disease_idx}')

# Class weights: inverse-frequency × asymmetric down-weighting healthy.
# WeightedRandomSampler убран — двойное взвешивание ломает обучение.
counts = [0] * num_classes
for _, y in train_ds.samples: counts[y] += 1
total = sum(counts)
inv_freq = [total / (num_classes * max(c, 1)) for c in counts]
if healthy_idx >= 0:
    inv_freq[healthy_idx] *= HEALTHY_DOWNWEIGHT
class_weights = torch.tensor(inv_freq, dtype=torch.float32, device=device)
print('Class weights:')
for n, c, w in zip(class_names, counts, inv_freq):
    print(f'  {n:<22} count={c:>4} weight={w:.3f}')
if total > 0 and min(counts) < 60:
    print(f'⚠ Smallest class has {min(counts)} train samples.')


def make_loader(ds, batch, shuffle=False):
    return DataLoader(ds, batch_size=batch, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      persistent_workers=PERSISTENT,
                      prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
                      drop_last=shuffle)

train_loader = make_loader(train_ds, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(val_ds,   BATCH_SIZE, shuffle=False)


def build_b0(nc):
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, nc)
    for name, p in m.named_parameters():
        p.requires_grad = name.startswith('classifier.')
    return m.to(memory_format=torch.channels_last).to(device)


net = build_b0(num_classes)


def _resume_b0(net_):
    if not RESUME_FROM_BACKUP: return False
    for name in ('best.pth', 'classifier.pth'):
        p = MODELS_OUT / name
        if not p.exists(): continue
        try:
            state = torch.load(str(p), map_location='cpu', weights_only=True)
            cur = net_.state_dict()
            sm = sum(1 for k, v in state.items() if k in cur and cur[k].shape == v.shape)
            if sm < len(cur) * 0.7:
                print(f'  ✗ {name}: arch mismatch ({sm}/{len(cur)})'); continue
            for hk in [k for k in state if k.startswith('classifier.') and k.endswith('.weight') and state[k].ndim == 2]:
                if state[hk].shape[0] != num_classes:
                    for k in list(state):
                        if k.startswith('classifier.'): del state[k]
                    print(f'  · {name}: head mismatch — features only.')
                    break
            net_.load_state_dict(state, strict=False)
            print(f'  ✓ Warm-start from {name}')
            return True
        except Exception as e:
            print(f'  ! resume {name}: {e}')
    return False

_resume_b0(net)

def _raw(m):
    return getattr(m, '_orig_mod', m)

def _maybe_compile(m):
    if not (USE_TORCH_COMPILE and hasattr(torch, 'compile') and device.type == 'cuda'):
        return m
    try:
        out = torch.compile(m, mode='reduce-overhead', dynamic=False)
        print('  ✓ torch.compile enabled (reduce-overhead)')
        return out
    except Exception as e:
        print(f'  ! torch.compile failed: {e}')
        return m


def _rand_bbox(w, h, lam):
    cr = math.sqrt(max(0.0, 1.0 - lam))
    cw, ch = int(w * cr), int(h * cr)
    cx, cy = random.randint(0, w - 1), random.randint(0, h - 1)
    return (max(0, cx-cw//2), max(0, cy-ch//2),
            min(w, cx+cw//2), min(h, cy+ch//2))

def _safe_perm(y):
    if not_a_plant_idx < 0:
        return torch.randperm(y.size(0), device=y.device)
    is_nap = (y == not_a_plant_idx)
    perm = torch.arange(y.size(0), device=y.device)
    pi = torch.where(~is_nap)[0]; ni = torch.where(is_nap)[0]
    if len(pi) > 1: perm[pi] = pi[torch.randperm(len(pi), device=y.device)]
    if len(ni) > 1: perm[ni] = ni[torch.randperm(len(ni), device=y.device)]
    return perm

def cutmix(x, y, alpha=1.0):
    lam = float(np.random.beta(alpha, alpha))
    perm = _safe_perm(y)
    x1, y1, x2, y2 = _rand_bbox(x.size(3), x.size(2), lam)
    x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam = 1.0 - ((x2-x1)*(y2-y1)) / (x.size(3)*x.size(2))
    return x, y, y[perm], lam

def mixup(x, y, alpha=MIXUP_ALPHA):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    perm = _safe_perm(y)
    x = lam * x + (1.0 - lam) * x[perm]
    return x, y, y[perm], lam


crit = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
use_amp = device.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if (use_amp and HAS_BF16) else torch.float16
USE_SCALER = use_amp and AMP_DTYPE is torch.float16
try:
    autocast_ctx = lambda: torch.amp.autocast('cuda', enabled=use_amp, dtype=AMP_DTYPE)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_SCALER)
except (AttributeError, TypeError):
    autocast_ctx = lambda: torch.cuda.amp.autocast(enabled=use_amp, dtype=AMP_DTYPE)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)
print(f'AMP: {AMP_DTYPE} | scaler={USE_SCALER} | batch={BATCH_SIZE} | workers={NUM_WORKERS}')


class ModelEMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        d = self.decay
        for ep, p in zip(self.ema.parameters(), model.parameters()):
            ep.mul_(d).add_(p.detach(), alpha=1.0 - d)
        for eb, b in zip(self.ema.buffers(), model.buffers()):
            eb.copy_(b)


def make_opt_sched(params, lr, epochs):
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    warm = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=1)
    cos = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs - 1, 1))
    return opt, optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[1])


def _tta(model, x):
    with autocast_ctx():
        p1 = F.softmax(model(x), dim=1)
        p2 = F.softmax(model(torch.flip(x, dims=[3])), dim=1)
    return (p1 + p2) * 0.5


def train_epoch(loader, opt, model, ema_, criterion):
    model.train()
    ls = cor = nt = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        r = random.random()
        cm = r < CUTMIX_PROB
        mu = (not cm) and r < CUTMIX_PROB + MIXUP_PROB
        mix = cm or mu
        if cm: x, ya, yb, lam = cutmix(x, y, 1.0)
        elif mu: x, ya, yb, lam = mixup(x, y, MIXUP_ALPHA)
        with autocast_ctx():
            logits = model(x)
            loss = lam*criterion(logits, ya) + (1-lam)*criterion(logits, yb) if mix else criterion(logits, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), GRAD_CLIP)
        scaler.step(opt); scaler.update()
        ema_.update(model)
        ls += loss.item() * x.size(0)
        cor += (logits.argmax(1) == y).sum().item()
        nt += x.size(0)
    return ls/nt, cor/nt

@torch.inference_mode()
def val_epoch(model, loader):
    """Returns (loss, acc, disease_f1, healthy_bias_ratio, all_preds, all_labels)."""
    model.eval()
    ls = cor = nt = 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        probs = _tta(model, x)
        ls += F.nll_loss(torch.log(probs.clamp_min(1e-8)), y).item() * x.size(0)
        pred = probs.argmax(1)
        cor += (pred == y).sum().item()
        nt += x.size(0)
        all_preds.append(pred.cpu())
        all_labels.append(y.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    f1d = (f1_score(all_labels, all_preds, labels=disease_idx,
                    average='macro', zero_division=0)
           if disease_idx else float('nan'))
    if healthy_idx >= 0:
        p_h = int((all_preds == healthy_idx).sum())
        t_h = int((all_labels == healthy_idx).sum())
        bias = p_h / max(t_h, 1)
    else:
        bias = float('nan')
    return ls/nt, cor/nt, f1d, bias, all_preds, all_labels


best_f1 = -1.0
no_imp = 0
log_path = MODELS_OUT / 'training_log.txt'

def _log(line):
    print(line)
    try:
        with log_path.open('a', encoding='utf-8') as f: f.write(line + '\n')
    except Exception: pass


def phase(lbl, epochs, lr, allow_es, model, ema_, criterion, tl, vl, ckpt='best.pth'):
    global best_f1, no_imp
    opt, sch = make_opt_sched([p for p in model.parameters() if p.requires_grad], lr, epochs)
    _log(f'\n=== {lbl} ===')
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a = train_epoch(tl, opt, model, ema_, criterion)
        vl_l, vl_a, vl_f1, vl_bias, _, _ = val_epoch(ema_.ema, vl)
        sch.step()
        bias_flag = '🚨' if vl_bias > 2.0 else ('⚠' if vl_bias > 1.5 else ' ')
        _log(f'  {ep}/{epochs} train_l={tr_l:.3f} acc={tr_a:.3f} | '
             f'val_l={vl_l:.3f} acc={vl_a:.3f} disease_F1={vl_f1:.3f} '
             f'h_bias={vl_bias:.2f}{bias_flag} ({time.time()-t0:.1f}s)')
        if vl_f1 > best_f1 + 1e-4:
            best_f1 = vl_f1
            torch.save(ema_.ema.state_dict(), MODELS_OUT / ckpt)
            no_imp = 0
        else:
            no_imp += 1
        if allow_es and no_imp >= EARLY_STOP:
            _log(f'  ⏸ Early stop (no F1 improvement for {EARLY_STOP} epochs)'); break


ema = ModelEMA(net, EMA_DECAY)
net = _maybe_compile(net)
phase('Phase A B0 head', 12, 1e-3 * _LR_SCALE, False, net, ema, crit, train_loader, val_loader)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

for name, p in _raw(net).named_parameters():
    p.requires_grad = name.startswith(('features.4', 'features.5', 'features.6', 'features.7', 'features.8', 'classifier'))
print(f'Phase B trainable: {sum(p.numel() for p in _raw(net).parameters() if p.requires_grad):,}')
no_imp = 0
phase('Phase B B0 fine-tune', 30, 5e-5 * _LR_SCALE, True, net, ema, crit, train_loader, val_loader)

best_state = torch.load(str(MODELS_OUT / 'best.pth'), map_location='cpu', weights_only=True)
ema.ema.load_state_dict(best_state); _raw(net).load_state_dict(best_state)
torch.save(best_state, MODELS_OUT / 'classifier.pth')
b0_best_f1 = best_f1
print(f'\nBest val disease F1: {b0_best_f1:.4f}')

# Final B0 val pass for temperature calibration + reporting
ema.ema.eval()
all_logits, all_y, all_pred = [], [], []
with torch.inference_mode():
    for x, y in val_loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        with autocast_ctx(): logits = ema.ema(x)
        probs = _tta(ema.ema, x)
        all_logits.append(logits.float().cpu())
        all_pred.append(probs.argmax(1).cpu())
        all_y.append(y)
        del logits, probs, x
all_logits = torch.cat(all_logits)
all_y = torch.cat(all_y)
all_pred = torch.cat(all_pred)

b0_best_acc = float((all_pred == all_y).float().mean())
disease_f1_final = float(f1_score(all_y.numpy(), all_pred.numpy(),
                                  labels=disease_idx, average='macro', zero_division=0))
print(f'Final val acc: {b0_best_acc:.4f}  disease F1: {disease_f1_final:.4f}')

# Healthy bias on final val
if healthy_idx >= 0:
    p_h = int((all_pred == healthy_idx).sum())
    t_h = int((all_y == healthy_idx).sum())
    healthy_bias_final = p_h / max(t_h, 1)
    print(f'Healthy bias: {healthy_bias_final:.2f} (norm <1.5, alarm >2.0)')
else:
    healthy_bias_final = float('nan')

# Temperature calibration
Ts = np.concatenate([np.linspace(0.3, 1.0, 36), np.linspace(1.05, 5.0, 80)])
best_T, best_nll = 1.0, float('inf')
for T in Ts:
    nll = F.cross_entropy(all_logits / T, all_y).item()
    if nll < best_nll: best_nll, best_T = nll, float(T)
raw_nll = F.cross_entropy(all_logits, all_y).item()
print(f'T = {best_T:.3f}  (NLL: {raw_nll:.4f} → {best_nll:.4f})')

(MODELS_OUT / 'temperature.json').write_text(json.dumps({
    'temperature': best_T, 'val_nll_raw': round(raw_nll, 4),
    'val_nll_calibrated': round(best_nll, 4),
}, indent=2), encoding='utf-8')

print('\n── Per-class metrics ──')
print(classification_report(all_y.tolist(), all_pred.tolist(),
                            target_names=class_names, digits=3, zero_division=0))
cm = confusion_matrix(all_y.tolist(), all_pred.tolist(), labels=list(range(num_classes)))
print('Confusion matrix:')
print('              ' + ' '.join(f'{n[:10]:>11}' for n in class_names))
for name, row in zip(class_names, cm):
    print(f'{name:>14}  ' + ' '.join(f'{v:>11}' for v in row))

del all_logits, all_y, all_pred, train_loader
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 7. ONNX export + INT8 + OOD-калибровка

Classifier ONNX FP32 opset=18 + dynamic INT8 на MatMul. Sanity-check Torch vs FP32 vs INT8 по disease F1 (drift INT8 ≤ 0.020). OOD-sweep `NAP_CLASSIFIER_MSP_FLOOR ∈ {0.35..0.60}`. Результат → `classes.json["ood_calibration"]`.

In [ ]:
import math
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_FP32 = MODELS_OUT / 'classifier.onnx'
ONNX_INT8 = MODELS_OUT / 'classifier_int8.onnx'


def _export(model, dst, size, label):
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, size, size, dtype=torch.float32)
    with torch.inference_mode():
        torch.onnx.export(m, dummy, str(dst), input_names=['input'], output_names=['logits'],
            dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
            opset_version=18, do_constant_folding=True, dynamo=False)
    del m, dummy; gc.collect()
    print(f'ONNX FP32 {label} → {dst.name} ({dst.stat().st_size/1e6:.1f} MB)')

    mo = onnx.load(str(dst))
    onnx.checker.check_model(mo)
    try:
        from onnxsim import simplify
        s, ok = simplify(mo)
        if ok: onnx.save(s, str(dst)); print(f'  + simplified')
    except Exception as e:
        print(f'  · onnxsim skip: {e}')
    del mo; gc.collect()


def _quant(src, dst, label):
    dst.unlink(missing_ok=True)
    quantize_dynamic(model_input=str(src), model_output=str(dst),
                     weight_type=QuantType.QInt8, op_types_to_quantize=['MatMul'], per_channel=True)
    print(f'ONNX INT8 {label} → {dst.name} ({dst.stat().st_size/1e6:.1f} MB)')


_export(ema.ema, ONNX_FP32, IMG_SIZE, 'B0')
_quant(ONNX_FP32, ONNX_INT8, 'B0')


def onnx_eval(path, loader):
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    cor = nt = 0
    preds_all, labels_all = [], []
    for x, y in loader:
        try: logits = sess.run(None, {inp: x.numpy().astype(np.float32, copy=False)})[0]
        except Exception as e: print(f'  ! {e}'); return float('nan'), float('nan')
        pred = np.argmax(logits, axis=1)
        cor += int((pred == y.numpy()).sum())
        nt += len(y)
        preds_all.append(pred); labels_all.append(y.numpy())
    del sess; gc.collect()
    if nt == 0: return float('nan'), float('nan')
    p = np.concatenate(preds_all); l = np.concatenate(labels_all)
    f1d = f1_score(l, p, labels=disease_idx, average='macro', zero_division=0) if disease_idx else float('nan')
    return cor/nt, f1d


print('\n── ONNX sanity check ──')
acc_fp32, f1_fp32 = onnx_eval(ONNX_FP32, val_loader)
acc_int8, f1_int8 = onnx_eval(ONNX_INT8, val_loader)
print(f'Torch EMA+TTA acc: {b0_best_acc:.4f}  disease_F1: {disease_f1_final:.4f}')
print(f'ONNX FP32        acc: {acc_fp32:.4f}  disease_F1: {f1_fp32:.4f}')
print(f'ONNX INT8        acc: {acc_int8:.4f}  disease_F1: {f1_int8:.4f}')
drift = abs(f1_int8 - f1_fp32)
print(f'  {"✓" if drift <= 0.020 else "⚠"} INT8 disease_F1 drift {drift:.3f}')


# OOD calibration
print('\n── OOD calibration sweep ──')
ood_neg = list((CLS_DIR / 'val' / 'not_a_plant').iterdir()) if (CLS_DIR / 'val' / 'not_a_plant').exists() else []
ood_pos = [Path(p) for p, y in val_ds.samples if class_names[y] != 'not_a_plant']
print(f'  pos={len(ood_pos)} neg={len(ood_neg)}')

if ood_neg and ood_pos:
    sess = ort.InferenceSession(str(ONNX_INT8), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    nap_idx = class_names.index('not_a_plant') if 'not_a_plant' in class_names else -1
    mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])

    def _msp(p):
        try:
            with Image.open(p) as im:
                im = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                arr = np.asarray(im, dtype=np.float32) / 255.0
                arr = ((arr - mean) / std).transpose(2, 0, 1)[None].astype(np.float32, copy=False)
            l = sess.run(None, {inp: arr})[0][0]
            l -= l.max(); pp = np.exp(l); pp /= pp.sum()
            return float(pp.max()), int(pp.argmax())
        except Exception:
            return 0.0, -1

    pos = [_msp(p) for p in ood_pos]
    neg = [_msp(p) for p in ood_neg]
    del sess; gc.collect()

    rows = []
    print('  floor | TNR(NAP) | FPR(plants)')
    for floor in (0.35, 0.40, 0.45, 0.50, 0.55, 0.60):
        tnr = sum(1 for m, a in neg if m < floor and a != nap_idx) / max(len(neg), 1)
        fpr = sum(1 for m, a in pos if m < floor and a != nap_idx) / max(len(pos), 1)
        rows.append((floor, tnr, fpr))
        print(f'   {floor:.2f}  |  {tnr*100:5.1f}%  |  {fpr*100:5.1f}%')
    best = max(rows, key=lambda r: r[1] - r[2])
    suggested_msp_floor = best[0]
    print(f'  ✓ Suggested MSP floor = {suggested_msp_floor:.2f}')
else:
    suggested_msp_floor = 0.45


classes_meta = {
    '_comment': 'PlantVision houseplant classifier — disease-F1 primary metric',
    'model_version': '1.2',
    'architecture': {
        'detector': 'YOLOv8n (single-class leaf)',
        'classifier': 'EfficientNet-B0 (IMAGENET1K_V1)',
    },
    'num_classes': num_classes,
    'class_names': class_names,
    'detector_classes': ['leaf'],
    'val_accuracy_torch_tta': round(b0_best_acc, 4),
    'val_disease_f1_torch_tta': round(disease_f1_final, 4),
    'val_accuracy_onnx_fp32': round(acc_fp32, 4) if not math.isnan(acc_fp32) else None,
    'val_accuracy_onnx_int8': round(acc_int8, 4) if not math.isnan(acc_int8) else None,
    'val_disease_f1_onnx_fp32': round(f1_fp32, 4) if not math.isnan(f1_fp32) else None,
    'val_disease_f1_onnx_int8': round(f1_int8, 4) if not math.isnan(f1_int8) else None,
    'healthy_bias_ratio': round(healthy_bias_final, 3) if not math.isnan(healthy_bias_final) else None,
    'temperature': round(best_T, 4),
    'training': {
        'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE,
        'cutmix_prob': CUTMIX_PROB, 'mixup_prob': MIXUP_PROB, 'mixup_alpha': MIXUP_ALPHA,
        'label_smoothing': LABEL_SMOOTHING, 'ema_decay': EMA_DECAY, 'grad_clip': GRAD_CLIP,
        'class_balanced_loss': True,
        'healthy_downweight': HEALTHY_DOWNWEIGHT,
        'sampler': 'natural distribution (no WeightedRandomSampler)',
        'primary_metric': 'disease_macro_f1',
        'tta': 'hflip',
        'augmentation': 'TrivialAugment + WarmLight + JPEG + RandomErasing + MotionBlur + RandomShadow',
        'hp_target': 800, 'hp_crops_per_photo': 1, 'hp_hard_aug_ratio': 0.30,
        'real_world_in_train': True,
        'calibration': 'temperature scaling on val NLL',
        'quantization': 'dynamic INT8 on MatMul',
    },
    'ood_calibration': {'suggested_classifier_msp_floor': float(suggested_msp_floor)},
    'proxy_trained_classes': ['bacterial_leaf_spot', 'late_blight', 'sunburn'],
    '_proxy_trained_note': 'Trained on non-houseplant visual proxies (PV tomato/pepper bacterial, PV potato/tomato late blight, strawberry leaf scorch ≈ sunburn). Live eval shows F1≈0 on these classes — server surfaces "proxy_trained:<class>" warning when they win.',
}
(MODELS_OUT / 'classes.json').write_text(
    json.dumps(classes_meta, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\nclasses.json → {MODELS_OUT / "classes.json"}')


## 8. End-to-end demo

Detector → crop → classifier (5-crop+hflip TTA, temperature-calibrated softmax).

In [ ]:
import matplotlib.pyplot as plt
from ultralytics import YOLO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
det = YOLO(str(MODELS_OUT / 'detector.pt'))
sess = ort.InferenceSession(str(MODELS_OUT / 'classifier_int8.onnx'), providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name

mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
TEMP = json.loads((MODELS_OUT / 'temperature.json').read_text(encoding='utf-8'))['temperature']

def _norm(arr):
    arr = arr.astype(np.float32) / 255.0
    return ((arr - mean) / std).transpose(2, 0, 1)[None]

def _five_crop(arr, size):
    H, W = arr.shape[:2]
    if H < size or W < size: return [arr]
    cy, cx = H // 2, W // 2
    return [arr[:size, :size], arr[:size, W-size:], arr[H-size:, :size], arr[H-size:, W-size:],
            arr[cy-size//2:cy+size//2, cx-size//2:cx+size//2]]

def _classify(crop_pil):
    arr = np.array(crop_pil.convert('RGB'))
    H, W = arr.shape[:2]
    if min(H, W) > IMG_SIZE * 1.5:
        crops = []
        for c in _five_crop(arr, IMG_SIZE):
            c = np.array(Image.fromarray(c).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
            crops.append(c); crops.append(c[:, ::-1, :])
        x = np.concatenate([_norm(c) for c in crops], axis=0).astype(np.float32, copy=False)
    else:
        a = np.array(crop_pil.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
        x = np.concatenate([_norm(a), _norm(a[:, ::-1, :])], axis=0).astype(np.float32, copy=False)
    logits = sess.run(None, {inp_name: x})[0]
    logits -= logits.max(axis=1, keepdims=True)
    p = np.exp(logits / max(TEMP, 1e-3)); p /= p.sum(axis=1, keepdims=True)
    return p.mean(axis=0)


demo = []
for cls in class_names[:6]:
    d = CLS_DIR / 'val' / cls
    if d.exists():
        for f in list(d.iterdir())[:1]: demo.append((cls, f))

cols = min(len(demo), 4)
rows = (len(demo) + cols - 1) // max(cols, 1)
plt.figure(figsize=(4*cols, 4*rows))
for i, (true_cls, path) in enumerate(demo, 1):
    img = Image.open(path).convert('RGB')
    res = det.predict(np.array(img), imgsz=640, conf=0.4, verbose=False, device=device)
    boxes = res[0].boxes.xyxy.cpu().numpy().tolist() if res and res[0].boxes is not None else []
    if boxes:
        x1, y1, x2, y2 = (int(round(v)) for v in boxes[0])
        crop = img.crop((x1, y1, x2, y2))
    else:
        crop = img
    p = _classify(crop)
    pred = class_names[int(np.argmax(p))]
    plt.subplot(rows, cols, i); plt.imshow(img)
    plt.title(f'true={true_cls}\npred={pred} ({float(p.max()):.2f})'); plt.axis('off')
plt.tight_layout(); plt.show()
del det; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 9. Артефакты

In [ ]:
artifacts = ['detector.pt', 'detector.onnx',
             'classifier.pth', 'classifier.onnx', 'classifier_int8.onnx',
             'classes.json', 'temperature.json', 'MODEL_CARD.md']
print(f'Артефакты в {MODELS_OUT}:')
for n in artifacts:
    p = MODELS_OUT / n
    if p.exists(): print(f'  {n:<28} {p.stat().st_size/1e6:>6.2f} MB')
    else: print(f'  {n:<28} (missing)')
print('\nКопировать в server/models/.')


## 10. MODEL_CARD.md

In [ ]:
from datetime import datetime, timezone

def _sz(p): return f'{p.stat().st_size/1e6:.2f} MB' if p.exists() else '—'

card = f'''# PlantVision — MODEL_CARD

Generated: {datetime.now(timezone.utc).isoformat(timespec="seconds")}

## Pipeline
Stage 1: YOLOv8n single-class leaf detector. Stage 2: EfficientNet-B0 @ {IMG_SIZE}.

## Classes
{", ".join(class_names)}

## Held-out val metrics

Primary metric: **disease macro-F1** (8 диагнозов, без `healthy` / `not_a_plant`).

| Path | top-1 | disease F1 |
|------|-------|------------|
| Torch B0 (EMA + h-flip TTA) | {b0_best_acc:.4f} | {disease_f1_final:.4f} |
| ONNX FP32 | {acc_fp32:.4f} | {f1_fp32:.4f} |
| ONNX INT8 | {acc_int8:.4f} | {f1_int8:.4f} |

Healthy bias (predicted_healthy / true_healthy): **{healthy_bias_final:.2f}** (norm <1.5, alarm >2.0)
Temperature: {best_T:.3f} (val NLL {raw_nll:.4f} → {best_nll:.4f})

## Artefact sizes
| File | Size |
|------|------|
| detector.pt | {_sz(MODELS_OUT / "detector.pt")} |
| detector.onnx | {_sz(MODELS_OUT / "detector.onnx")} |
| classifier.pth | {_sz(MODELS_OUT / "classifier.pth")} |
| classifier_int8.onnx | {_sz(MODELS_OUT / "classifier_int8.onnx")} |

## Config
- IMG_SIZE: {IMG_SIZE}
- BATCH_SIZE: {BATCH_SIZE}
- Loss: class-balanced CE (inverse-freq × healthy {HEALTHY_DOWNWEIGHT}), label_smoothing={LABEL_SMOOTHING}
- CutMix / MixUp: {CUTMIX_PROB} / {MIXUP_PROB}
- EMA decay: {EMA_DECAY}
- Sampler: natural (no WeightedRandomSampler)
- HP target: 800 photos, 1 crop/photo, 30% with hard-aug (blur + low-quality JPEG)
- Real-world фото: материализованы из RAW/real_world/ (80/20 split)
- Suggested NAP_CLASSIFIER_MSP_FLOOR: {suggested_msp_floor:.2f}

## Known limitations
- `mosaic_virus` ↔ `iron_chlorosis` пересекаются на жёлтых листьях.
- `late_blight` proxy — field-crop, transfer на indoor ornamentals имперфектный.
- `bacterial_leaf_spot` / `sunburn` обучены через визуальные proxy.
- `anthracnose` слабый класс — мало iNat данных, добавь real-world фото в `RAW/real_world/anthracnose/`.
'''

(MODELS_OUT / 'MODEL_CARD.md').write_text(card, encoding='utf-8')
print(f'MODEL_CARD.md → {MODELS_OUT / "MODEL_CARD.md"}')
